# Agentic Memory (A-MEM / Zettelkasten-style) | Agent Memory System

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from typing import Dict, List, Set
from dataclasses import dataclass, field
from datetime import datetime

set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [2]:
model = ChatOpenAI(model="gpt-4o")

In [3]:
# Zettelkasten-style Agentic Memory
# Production A-MEM uses vector embeddings for semantic search across notes (not just tag lookup)

@dataclass
class Note:
    note_id: str
    content: str
    tags: List[str]
    links: Set[str] = field(default_factory=set)

class ZettelkastenMemory:
    def __init__(self):
        self.notes: Dict[str, Note] = {}
        self._next_id = 1

    def add(self, content: str, tags: List[str], links: List[str] = None) -> str:
        nid = f"note-{self._next_id}"
        self._next_id += 1
        note = Note(note_id=nid, content=content, tags=tags, links=set(links or []))
        self.notes[nid] = note
        for lid in note.links:
            if lid in self.notes:
                self.notes[lid].links.add(nid)
        return nid

    def search_by_tag(self, tag: str) -> List[Note]:
        return [n for n in self.notes.values() if tag in n.tags]

    def traverse(self, start_id: str, depth: int = 2) -> List[Note]:
        """Follow links from a starting note to discover connected knowledge."""
        visited, queue = set(), [(start_id, 0)]
        result = []
        while queue:
            nid, d = queue.pop(0)
            if nid in visited or d > depth: continue
            visited.add(nid)
            if nid in self.notes:
                result.append(self.notes[nid])
                for link in self.notes[nid].links:
                    queue.append((link, d + 1))
        return result

    def get_context(self, tags: List[str], max_notes: int = 5) -> str:
        relevant = []
        for tag in tags:
            relevant.extend(self.search_by_tag(tag))
        seen = set()
        unique = []
        for n in relevant:
            if n.note_id not in seen:
                seen.add(n.note_id)
                unique.append(n)
        return "\n\n".join(f"[{n.note_id}] ({', '.join(n.tags)}): {n.content}" for n in unique[:max_notes])

In [4]:
zk = ZettelkastenMemory()
n1 = zk.add("LLM agents perform better with structured JSON output", ["llm", "agents"])
n2 = zk.add("JSON mode reduces parsing errors by 90%", ["llm", "json"], links=[n1])
n3 = zk.add("Agent memory should support both semantic search and metadata queries", ["agents", "memory"], links=[n1])

context = zk.get_context(["agents", "memory"])
response = model.invoke(
    f"Using your knowledge notes, answer: What's the best approach to agent memory?\n\n"
    f"Notes:\n{context}"
)
print(response.content)

# --- Graph traversal: follow links from a starting note ---
connected = zk.traverse(n1, depth=2)
print("\nGraph traversal from n1 (depth=2):")
for note in connected:
    linked_to = ", ".join(note.links) if note.links else "none"
    print(f"  {note.note_id}: {note.content[:60]}... -> links: [{linked_to}]")

The best approach to agent memory, based on the provided notes, is to ensure that it supports both semantic search and metadata queries. This allows the agent to effectively retrieve and utilize relevant information based on the content (semantic search) and specific attributes or tags (metadata queries) of the information stored. Integrating these capabilities ensures the agent can access and apply past interactions or knowledge efficiently, enhancing its performance and accuracy in various tasks.

Graph traversal from n1 (depth=2):
  note-1: LLM agents perform better with structured JSON output... -> links: [note-3, note-2]
  note-3: Agent memory should support both semantic search and metadat... -> links: [note-1]
  note-2: JSON mode reduces parsing errors by 90%... -> links: [note-1]
